# Define a Knowledge Graph from Scratch

End-to-end walkthrough of the genai-graph new design:

1. Define Pydantic domain models
2. Declare `GraphNode` / `GraphRelation` / `GraphSchema`
3. Inspect the compiled schema
4. Visualise the **schema** as an interactive HTML diagram
5. Ingest data into an in-memory Ladybug graph
6. Query the KG with Cypher
7. Visualise the **KG data** as an interactive HTML diagram

No external database or config required — runs fully in memory.


In [ ]:
# -- Shared display helper import ---------------------------------------------
# -- Set log level to INFO (suppress DEBUG traces) ----------------------------
import sys

from loguru import logger

from genai_graph.utils.notebooks import show_html_in_notebook

logger.remove()
logger.add(sys.stderr, level="INFO")

## 1. Define Your Domain Models

Plain Pydantic models — no graph imports needed here.

In [ ]:
from pydantic import BaseModel, Field


class Address(BaseModel):
    city: str
    country: str = "Unknown"


class Company(BaseModel):
    name: str
    sector: str | None = None
    hq: Address | None = None


class Person(BaseModel):
    name: str
    role: str | None = None


class Risk(BaseModel):
    description: str
    impact: str = "medium"


class Project(BaseModel):
    """Root model — the entry point for graph traversal."""

    title: str
    status: str = "active"
    client: Company
    team: list[Person] = Field(default_factory=list)
    risks: list[Risk] = Field(default_factory=list)


print("Models defined:", [m.__name__ for m in [Project, Company, Person, Risk, Address]])

## 2. Declare the Schema

Wrap each model in a `GraphNode` and specify identity fields.
Nested models become relation endpoints — `GraphRelation` names the edge.


In [ ]:
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema

project_node = GraphNode(node_class=Project, name_from="title", key_from="title", description="A project review")
company_node = GraphNode(node_class=Company, name_from="name", key_from="name", description="A client company")
person_node = GraphNode(node_class=Person, name_from="name", key_from="name", description="A team member")
risk_node = GraphNode(node_class=Risk, name_from="description", key_from="AUTO_ID", description="A project risk")

schema = GraphSchema(
    root_model_class=Project,
    nodes=[project_node, company_node, person_node, risk_node],
    relations=[
        GraphRelation(from_node=project_node, to_node=company_node, name="FOR_CLIENT"),
        GraphRelation(from_node=project_node, to_node=person_node, name="HAS_MEMBER"),
        GraphRelation(from_node=project_node, to_node=risk_node, name="HAS_RISK"),
    ],
)
print(f"Schema: {len(schema.nodes)} nodes, {len(schema.relations)} relations")

## 3. Inspect the Compiled Schema

`GraphSchema` auto-deduces field paths and excluded fields at construction time.

In [ ]:
print("=== Nodes ===")
for n in schema.nodes:
    fp = n.field_paths
    ex = n.excluded_fields
    print(f"  {n.label:15}  field_paths={fp!r}  excluded={ex!r}")

print("\n=== Relations ===")
for r in schema.relations:
    print(f"  {r.from_node.label} -[{r.name}]-> {r.to_node.label}")

print("\n=== Markdown table ===")
from genai_graph.kg.schema import ResolvedSchema

resolved = ResolvedSchema.from_graph_schema(schema)
print(resolved.to_markdown())

print("\n=== Validation warnings ===")
from genai_graph.kg.schema.compiler import validate_schema_coherence

warnings = validate_schema_coherence(schema)
print(warnings if warnings else "None — schema is clean")

## 4. Visualise the Schema

Interactive D3.js diagram of node types and relationship types.

In [ ]:
schema_html = resolved.to_html()
show_html_in_notebook(schema_html, "schema")
print(f"Schema HTML: {len(schema_html):,} bytes")

## 5. Ingest Data

`create_graph()` walks the root model, splits fields into nodes and relations,
and upserts everything into the graph database.


In [ ]:
from genai_graph.kg.ingest import create_graph, restart_database

backend = restart_database()  # fresh in-memory Ladybug DB

sample_projects = [
    Project(
        title="Cloud Migration",
        status="in-progress",
        client=Company(name="Acme Corp", sector="Retail", hq=Address(city="Paris")),
        team=[
            Person(name="Alice Martin", role="Lead"),
            Person(name="Bob Chen", role="Engineer"),
        ],
        risks=[
            Risk(description="Data loss during migration", impact="high"),
            Risk(description="Timeline overrun", impact="medium"),
        ],
    ),
    Project(
        title="ERP Modernisation",
        status="planning",
        client=Company(name="GlobalSoft", sector="Finance"),
        team=[Person(name="Alice Martin", role="Lead"), Person(name="Carol Li", role="Architect")],
        risks=[Risk(description="Budget overrun", impact="high")],
    ),
    Project(
        title="Data Platform",
        status="active",
        client=Company(name="Acme Corp", sector="Retail", hq=Address(city="Paris")),
        team=[Person(name="Bob Chen", role="Engineer")],
        risks=[],
    ),
]

for project in sample_projects:
    create_graph(backend, project, schema)

print(f"Ingested {len(sample_projects)} projects")
for label in ["Project", "Company", "Person", "Risk"]:
    df = backend.execute_get_as_df(f"MATCH (n:{label}) RETURN count(n) AS cnt")
    print(f"  {label}: {df['cnt'].iloc[0]}")

## 6. Query the KG

Standard Cypher — identical syntax to Neo4j.

In [ ]:
import pandas as pd


def run(cypher: str, title: str = "") -> None:
    df = backend.execute_get_as_df(cypher)
    if title:
        print(f"\n--- {title} ---")
    print(df.to_string(index=False) if not df.empty else "(no results)")


# Projects and their clients
run(
    "MATCH (p:Project)-[:FOR_CLIENT]->(c:Company) RETURN p.title, p.status, c.name AS client",
    "Projects by client",
)

# Team members per project
run(
    "MATCH (p:Project)-[:HAS_MEMBER]->(m:Person) RETURN p.title, m.name, m.role ORDER BY p.title",
    "Team members",
)

# High-impact risks
run(
    "MATCH (p:Project)-[:HAS_RISK]->(r:Risk) WHERE r.impact = 'high' RETURN p.title, r.description",
    "High-impact risks",
)

# Persons involved in multiple projects
run(
    """
    MATCH (p:Project)-[:HAS_MEMBER]->(m:Person)
    WITH m.name AS person, count(DISTINCT p) AS projects
    WHERE projects > 1
    RETURN person, projects ORDER BY projects DESC
    """,
    "Persons in multiple projects",
)

## 7. Visualise the KG Data

Interactive D3.js graph of the actual nodes and edges in the database.

In [ ]:
from genai_graph.kg.export.html import generate_html

kg_html = generate_html(connection=backend)
show_html_in_notebook(kg_html, "kg_data")
print(f"KG HTML: {len(kg_html):,} bytes, ~{kg_html.count('"id"')} nodes")

## 8. Advanced Schema Features

The next cells illustrate four additional schema patterns:

| Feature | What it does |
|---|---|
| **`p_` convention** | Fields named `p_<prop>_` on the *to-node* class become edge properties |
| **Object embeddings** | `extra_classes` embeds a nested Pydantic model as a STRUCT property on the node |
| **Lambda `name_from`** | Pass a callable to build node names dynamically from multiple fields |
| **Field-value embeddings** | `index_fields` triggers embedding computation; use `embeddings_768@fake` in tests |


### 8a. `p_` Convention — Properties on Edges

Fields named `p_<prop>_` (prefixed `p_`, suffixed `_`) on the *to-node* Pydantic class
are automatically **promoted to edge properties** and removed from the node itself.

Here `Person` gets a `p_role_` field.  The `HAS_MEMBER` edge will carry a `role` property
instead of the node storing it directly.


In [ ]:
from pydantic import BaseModel, Field
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema
from genai_graph.kg.ingest import create_graph, restart_database


# --- Domain models -----------------------------------------------------------


class PersonV2(BaseModel):
    """Team member.  'p_role_' will become an edge property, not a node column."""

    name: str
    p_role_: str | None = None  # ← edge property convention


class CompanyV2(BaseModel):
    name: str
    sector: str | None = None


class ProjectV2(BaseModel):
    title: str
    status: str = "active"
    client: CompanyV2
    team: list[PersonV2] = Field(default_factory=list)


# --- Schema ------------------------------------------------------------------

project_v2 = GraphNode(node_class=ProjectV2, name_from="title", key_from="title")
company_v2 = GraphNode(node_class=CompanyV2, name_from="name", key_from="name")
person_v2 = GraphNode(node_class=PersonV2, name_from="name", key_from="name")

schema_v2 = GraphSchema(
    root_model_class=ProjectV2,
    nodes=[project_v2, company_v2, person_v2],
    relations=[
        GraphRelation(from_node=project_v2, to_node=company_v2, name="FOR_CLIENT"),
        # HAS_MEMBER edge will automatically carry a 'role' property (from p_role_)
        GraphRelation(from_node=project_v2, to_node=person_v2, name="HAS_MEMBER"),
    ],
)

# --- Ingest sample data ------------------------------------------------------

backend_v2 = restart_database()

data = ProjectV2(
    title="Alpha",
    status="active",
    client=CompanyV2(name="Acme", sector="Tech"),
    team=[
        PersonV2(name="Alice", p_role_="Lead"),
        PersonV2(name="Bob", p_role_="Engineer"),
    ],
)
create_graph(backend_v2, data, schema_v2)

# --- Query edge properties ---------------------------------------------------

df = backend_v2.execute_get_as_df(
    "MATCH (p:ProjectV2)-[r:HAS_MEMBER]->(m:PersonV2) RETURN p.title, m.name, r.role ORDER BY m.name"
)
print("Edge properties on HAS_MEMBER:")
print(df.to_string(index=False))

### 8b. Object Embeddings in a Node — `extra_classes`

Pass a nested Pydantic model to `extra_classes` on a `GraphNode`.
Instead of creating a separate node + edge, the nested object is stored as a
**STRUCT property** directly on the parent node.

Here `HqInfo` (city + country) is embedded inside `CompanyV3` rather than
becoming its own graph node.


In [ ]:
from pydantic import BaseModel, Field
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema
from genai_graph.kg.ingest import create_graph, restart_database


# --- Domain models -----------------------------------------------------------


class HqInfo(BaseModel):
    """Embedded struct — will be stored as a STRUCT column on CompanyV3."""

    city: str
    country: str = "Unknown"


class CompanyV3(BaseModel):
    name: str
    sector: str | None = None
    hq: HqInfo | None = None  # ← will be embedded as a STRUCT property


class ProjectV3(BaseModel):
    title: str
    client: CompanyV3


# --- Schema: HqInfo in extra_classes → embedded STRUCT, not a separate node --

company_v3 = GraphNode(
    node_class=CompanyV3,
    name_from="name",
    key_from="name",
    extra_classes=[HqInfo],  # ← embed HqInfo as a STRUCT on the node
)
project_v3 = GraphNode(node_class=ProjectV3, name_from="title", key_from="title")

schema_v3 = GraphSchema(
    root_model_class=ProjectV3,
    nodes=[project_v3, company_v3],
    relations=[GraphRelation(from_node=project_v3, to_node=company_v3, name="FOR_CLIENT")],
)

# --- Ingest ------------------------------------------------------------------

backend_v3 = restart_database()

for proj in [
    ProjectV3(title="Gamma", client=CompanyV3(name="Acme", sector="Retail", hq=HqInfo(city="Paris", country="France"))),
    ProjectV3(
        title="Delta", client=CompanyV3(name="GlobalSoft", sector="Finance", hq=HqInfo(city="London", country="UK"))
    ),
]:
    create_graph(backend_v3, proj, schema_v3)

# --- Query the embedded struct -----------------------------------------------

df = backend_v3.execute_get_as_df("MATCH (c:CompanyV3) RETURN c.name, c.sector, c.hq")
print("CompanyV3 nodes with embedded HqInfo struct:")
print(df.to_string(index=False))

### 8c. Lambda `name_from` — Dynamic Node Labels

`name_from` accepts any callable `(data: dict, node_type: str) -> str`.
This is useful when a single field is not enough to produce a unique, human-readable label.

Here we combine `first_name` + `last_name` into the node display name.


In [ ]:
from pydantic import BaseModel, Field
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema
from genai_graph.kg.ingest import create_graph, restart_database


# --- Domain models -----------------------------------------------------------


class Employee(BaseModel):
    first_name: str
    last_name: str
    department: str | None = None


class OrgUnit(BaseModel):
    code: str
    members: list[Employee] = Field(default_factory=list)


# --- Lambda name_from: combine first_name + last_name ------------------------


def employee_label(data: dict, node_type: str) -> str:
    """Build a display label from first + last name."""
    return f"{data.get('first_name', '')} {data.get('last_name', '')}".strip()


employee_node = GraphNode(
    node_class=Employee,
    name_from=employee_label,  # ← callable instead of a field name
    key_from="last_name",  # primary key is last_name (simple demo)
)
org_node = GraphNode(node_class=OrgUnit, name_from="code", key_from="code")

schema_org = GraphSchema(
    root_model_class=OrgUnit,
    nodes=[org_node, employee_node],
    relations=[GraphRelation(from_node=org_node, to_node=employee_node, name="EMPLOYS")],
)

# --- Ingest ------------------------------------------------------------------

backend_org = restart_database()

unit = OrgUnit(
    code="ENG",
    members=[
        Employee(first_name="Alice", last_name="Martin", department="Backend"),
        Employee(first_name="Bob", last_name="Chen", department="Frontend"),
        Employee(first_name="Carol", last_name="Li", department="Data"),
    ],
)
create_graph(backend_org, unit, schema_org)

# --- Query: show computed node names -----------------------------------------

df = backend_org.execute_get_as_df(
    "MATCH (e:Employee) RETURN e.name AS display_name, e.last_name, e.department ORDER BY e.name"
)
print("Employee nodes (name built by lambda):")
print(df.to_string(index=False))

### 8d. Field-Value Embeddings — `index_fields`

Set `index_fields` on a `GraphNode` to automatically compute vector embeddings
for one or more text fields during ingestion.

Each entry is either:
- a plain field name `"description"` → uses the default embeddings model from config
- a tuple `("description", "embeddings_768@fake")` → uses a specific model

The resulting column is named `<field>_embedding` (e.g. `description_embedding`)
and stored as `FLOAT[N]` so Ladybug can build a vector index on it.

> **Note:** `embeddings_768@fake` produces deterministic random 768-d vectors —
> perfect for development and testing without requiring an API key.


In [ ]:
from pydantic import BaseModel, Field

from genai_graph.kg.ingest import create_graph, restart_database
from genai_graph.kg.schema import GraphNode, GraphRelation, GraphSchema

# --- Domain models -----------------------------------------------------------


class Article(BaseModel):
    slug: str
    title: str
    summary: str


class Publication(BaseModel):
    name: str
    articles: list[Article] = Field(default_factory=list)


# --- Schema: embed 'title' and 'summary' fields with the fake model ----------

FAKE_EMBED_MODEL = "embeddings_768@fake"  # 768-d random vectors, no API key needed

article_node = GraphNode(
    node_class=Article,
    name_from="title",
    key_from="slug",
    index_fields=[
        ("title", FAKE_EMBED_MODEL),  # → title_embedding FLOAT[768]
        ("summary", FAKE_EMBED_MODEL),  # → summary_embedding FLOAT[768]
    ],
)
pub_node = GraphNode(node_class=Publication, name_from="name", key_from="name")

schema_emb = GraphSchema(
    root_model_class=Publication,
    nodes=[pub_node, article_node],
    relations=[GraphRelation(from_node=pub_node, to_node=article_node, name="CONTAINS")],
)

# --- Ingest ------------------------------------------------------------------

backend_emb = restart_database()

pub = Publication(
    name="Tech Weekly",
    articles=[
        Article(slug="ai-trends", title="AI Trends 2026", summary="A review of the latest AI advances."),
        Article(slug="cloud-costs", title="Taming Cloud Costs", summary="Strategies to reduce cloud spending."),
        Article(
            slug="vector-dbs", title="Vector Databases Explained", summary="How vector stores power semantic search."
        ),
    ],
)
create_graph(backend_emb, pub, schema_emb)

# --- Verify embeddings were stored -------------------------------------------
# Check that embedding vectors were persisted (retrieve first element of each vector)

df = backend_emb.execute_get_as_df(
    "MATCH (a:Article) RETURN a.slug, a.title, a.title_embedding[1] AS title_emb_dim0, a.summary_embedding[1] AS summary_emb_dim0"
)
print("Articles with first embedding component (confirms 768-d vectors were stored):")
print(df.to_string(index=False))

# --- Cosine similarity between two articles (native Ladybug vector ops) ------
# --- NOTE : use CALL QUERY_VECTOR_INDEX  for better performance on large datasets
sim_df = backend_emb.execute_get_as_df("""
    MATCH (a1:Article {slug: 'ai-trends'}), (a2:Article {slug: 'vector-dbs'})
    RETURN array_cosine_similarity(a1.summary_embedding, a2.summary_embedding) AS cosine_sim
""")
print("\nCosine similarity (ai-trends vs vector-dbs):", round(sim_df["cosine_sim"].iloc[0], 4))